# Experiment 3: Linear Regression and Regularization (Ridge, Lasso, ElasticNet)
This standalone notebook implements Linear Regression alongside Ridge (L2), Lasso (L1), and ElasticNet regularization on the Loan Amount prediction dataset.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex3."""
    if os.path.basename(os.getcwd()) == 'Ex3':
        if rel_path.startswith('Ex3/'):
            return rel_path[len('Ex3/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex3/plots'), exist_ok=True)

In [2]:
def run_experiment_3(train_path="Datasets/Loan_Amount_Dataset/loan-train.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 3: REGULARIZED REGRESSION ===")
    print("="*60)
    
    path = resolve_path(train_path)
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    
    target_col = 'LoanAmount' if 'LoanAmount' in df.columns else df.select_dtypes(include=[np.number]).columns[-1]
    df = df.dropna(subset=[target_col])
    
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())
        
    cat_cols = df.select_dtypes(include=['object']).columns
    df = pd.get_dummies(df, columns=[c for c in cat_cols if c != 'Loan_ID'], drop_first=True)
    if 'Loan_ID' in df.columns:
        df = df.drop(columns=['Loan_ID'])
        
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge (L2)': Ridge(alpha=1.0),
        'Lasso (L1)': Lasso(alpha=0.1),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5)
    }
    
    results = {}
    for name, model in models.items():
        model.fit(X_tr_sc, y_tr)
        y_pred = model.predict(X_te_sc)
        r2 = r2_score(y_te, y_pred)
        mse = mean_squared_error(y_te, y_pred)
        mae = mean_absolute_error(y_te, y_pred)
        results[name] = {
            'R2 Score': round(r2, 4),
            'RMSE': round(np.sqrt(mse), 2),
            'MAE': round(mae, 2)
        }
        
    print("\n=== EXPERIMENT 3 PIPELINE COMPLETE ===")
    return results

In [3]:
# Master Execution Cell
ex3_output = run_experiment_3()
display(pd.DataFrame(ex3_output).T.style.background_gradient(cmap='Greens', subset=['R2 Score']))

=== LAUNCHING EXPERIMENT 3: REGULARIZED REGRESSION ===

=== EXPERIMENT 3 PIPELINE COMPLETE ===


,R2 Score,RMSE,MAE
Linear Regression,0.093400,57.810000,37.010000
Ridge (L2),0.094100,57.790000,37.010000
Lasso (L1),0.095300,57.750000,36.950000
ElasticNet,0.109400,57.300000,37.030000
